# Understanding HOG: Histogram of Oriented Gradients

*A complete guide to feature descriptors for object detection*

---

**HOG (Histogram of Oriented Gradients)** is a powerful feature descriptor widely used for object detection, particularly famous for pedestrian detection as introduced by Dalal & Triggs in 2005. Unlike keypoint-based methods like SIFT, HOG creates a dense description of an entire image window, capturing local shape information through gradient distributions.

This notebook provides a detailed walkthrough of the HOG algorithm with mathematical foundations and visual examples.

## Table of Contents

1. [Overview](#overview)
2. [Step 1: Preprocessing](#step-1-preprocessing)
3. [Step 2: Gradient Computation](#step-2-gradient-computation)
4. [Step 3: Cell Histograms](#step-3-cell-histograms)
5. [Step 4: Block Normalization](#step-4-block-normalization)
6. [Complete Pipeline Summary](#complete-pipeline-summary)
7. [HOG vs SIFT Comparison](#comparison-hog-vs-sift)

## Overview

HOG transforms an image into a feature vector that captures local shape and appearance information:

| Step | Description | Output |
|------|-------------|--------|
| 1 | Preprocessing (Grayscale + Gamma) | 640×480 grayscale image |
| 2 | Gradient Computation | 640×480 magnitude + direction |
| 3 | Cell Histograms | 80×60 = 4800 cells × 9 bins |
| 4 | Block Normalization | 79×59 = 4661 blocks × 36 values |
| 5 | Feature Vector | **167,796-D descriptor** |

![HOG Complete Pipeline](images/hog_complete_summary.png)

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from PIL import Image
from IPython.display import display, Markdown
import os

# For better plots
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print("Setup complete!")

## Load Input Image

In [ ]:
def load_image():
    """Load or create a test image"""
    # Create a synthetic pedestrian-like image for demo
    img = np.random.rand(128, 64) * 0.3 + 0.3
    
    # Body silhouette
    img[20:110, 22:42] = 0.4  # body
    img[25:45, 25:39] = 0.6   # head
    img[50:100, 20:44] = 0.5  # torso
    img[100:120, 18:30] = 0.45  # left leg
    img[100:120, 34:46] = 0.45  # right leg
    
    return img

img = load_image()
H, W = img.shape

plt.figure(figsize=(6, 12))
plt.imshow(img, cmap='gray')
plt.title(f'Input Image: {W} × {H} pixels', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f"Image shape: {img.shape}")

---
# Step 1: Preprocessing

**Why?** Normalize image intensity to reduce lighting effects.

## Step 1.1: Grayscale Conversion

Convert color images to grayscale:

$$I(x,y) = 0.299 \times R(x,y) + 0.587 \times G(x,y) + 0.114 \times B(x,y)$$

**Why these weights?**
- Human eye is most sensitive to green, least to blue
- Standard ITU-R BT.601 coefficients

**Example:**

```
RGB values: R = 180, G = 120, B = 90

I(x,y) = 0.299 × 180 + 0.587 × 120 + 0.114 × 90
       = 53.82 + 70.44 + 10.26
       = 134.52

Normalized: 134.52 / 255 = 0.527
```

![Step 1.1: Grayscale](images/hog_real_step1_1_grayscale.png)

![RGB Channels](images/hog_real_step1_rgb_channels.png)

## Step 1.2: Gamma Correction

$$I_{corrected}(x,y) = I(x,y)^\gamma$$

where γ = 0.5 (square root) is commonly used.

**Example:**

```
Original pixels:
  I = 0.16  →  I_γ = 0.16^0.5 = 0.40
  I = 0.25  →  I_γ = 0.25^0.5 = 0.50
  I = 0.36  →  I_γ = 0.36^0.5 = 0.60
  I = 0.49  →  I_γ = 0.49^0.5 = 0.70

Effect: Dark regions enhanced, bright regions compressed.
```

![Step 1.2: Gamma](images/hog_real_step1_2_gamma.png)

![Gamma Comparison](images/hog_real_step1_gamma_comparison.png)

In [ ]:
def gamma_correction(img, gamma=0.5):
    """Apply gamma correction: I_corrected = I^gamma"""
    return np.power(img + 1e-8, gamma)

# Apply preprocessing
img_preprocessed = gamma_correction(img, gamma=0.5)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_preprocessed, cmap='gray')
axes[1].set_title('After Gamma Correction (γ=0.5)', fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Step 1: Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Step 2: Gradient Computation

**Why?** Gradients capture edge information—the boundaries of objects.

### Understanding the 3×3 Neighborhood

```
Pixel at (x, y):

       x-1    x    x+1
      ┌─────┬─────┬─────┐
y-1   │ NW  │  N  │ NE  │
      ├─────┼─────┼─────┤
y     │  W  │  C  │  E  │   C = center pixel
      ├─────┼─────┼─────┤
y+1   │ SW  │  S  │ SE  │
      └─────┴─────┴─────┘

For HOG gradients:
  - Gx uses W and E (horizontal neighbors)
  - Gy uses N and S (vertical neighbors)
```

## Step 2.1: Horizontal Gradient (Gx)

**Formula (Central Difference):**

$$G_x(x,y) = I(x+1, y) - I(x-1, y) = E - W$$

**Kernel:** `[-1, 0, +1]`

**Detects:** Vertical edges (intensity changes left-to-right)

**Example:**

```
Image patch around (150, 200):

       x=149  x=150  x=151
      ┌──────┬──────┬──────┐
y=200 │ 0.25 │ 0.50 │ 0.75 │   ← Strong horizontal gradient!
      └──────┴──────┴──────┘

Gx(150, 200) = 0.75 - 0.25 = 0.50   ← Strong positive
```

![Step 2.1: Gx](images/hog_real_step2_1_gx.png)

## Step 2.2: Vertical Gradient (Gy)

**Formula:**

$$G_y(x,y) = I(x, y+1) - I(x, y-1) = S - N$$

**Kernel:**
```
[-1]
[ 0]
[+1]
```

**Detects:** Horizontal edges (intensity changes top-to-bottom)

![Step 2.2: Gy](images/hog_real_step2_2_gy.png)

![Gx Gy Combined](images/hog_real_step2_gx_gy_combined.png)

## Step 2.3: Gradient Magnitude

**Formula:**

$$M(x,y) = \sqrt{G_x(x,y)^2 + G_y(x,y)^2}$$

**Interpretation:**
- Large M → Strong edge
- Small M → Flat region

**Example:**

```
Given: Gx = 0.50, Gy = 0.20

M(150, 200) = √(0.50² + 0.20²)
            = √(0.25 + 0.04)
            = √0.29
            = 0.539   ← Moderate-strong edge
```

![Step 2.3: Magnitude](images/hog_real_step2_3_magnitude.png)

## Step 2.4: Gradient Direction (Unsigned)

**Formula:**

$$\theta(x,y) = \arctan\left(\frac{G_y}{G_x}\right) \mod 180°$$

**Why unsigned (0° to 180°)?**
- A vertical dark-to-light edge (→) has θ = 0°
- A vertical light-to-dark edge (←) has θ = 180°
- Both are the SAME vertical edge! We want them in the same bin.

**Example:**

```
Given: Gx = 0.50, Gy = 0.20

θ(150, 200) = arctan(0.20 / 0.50)
            = arctan(0.40)
            = 21.8°  (nearly horizontal edge)
```

![Step 2.4: Direction](images/hog_real_step2_4_direction.png)

![Step 2.5: Gradient Vectors](images/hog_real_step2_5_vectors.png)

In [ ]:
def compute_gradients(img):
    """
    Compute gradient magnitude and direction.
    
    Gx = I(x+1, y) - I(x-1, y)  (horizontal)
    Gy = I(x, y+1) - I(x, y-1)  (vertical)
    M = √(Gx² + Gy²)            (magnitude)
    θ = arctan(Gy/Gx) mod 180°  (direction, unsigned)
    """
    h, w = img.shape
    gx = np.zeros_like(img)
    gy = np.zeros_like(img)
    
    # Horizontal gradient: I(x+1) - I(x-1)
    gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
    
    # Vertical gradient: I(y+1) - I(y-1)
    gy[1:-1, :] = img[2:, :] - img[:-2, :]
    
    # Magnitude
    magnitude = np.sqrt(gx**2 + gy**2)
    
    # Direction (unsigned: 0-180 degrees)
    direction = np.arctan2(gy, gx) * 180 / np.pi
    direction = direction % 180  # Convert to unsigned
    
    return magnitude, direction, gx, gy

magnitude, direction, gx, gy = compute_gradients(img_preprocessed)

print(f"Gradient magnitude range: [{magnitude.min():.4f}, {magnitude.max():.4f}]")
print(f"Gradient direction range: [{direction.min():.1f}°, {direction.max():.1f}°]")

---
# Step 3: Cell Histograms

## Step 3.1: Divide Image into 8×8 Cells

```
Image: 640 × 480 pixels
Cell:  8 × 8 pixels

Cells per row:    640 ÷ 8 = 80 cells
Cells per column: 480 ÷ 8 = 60 cells
Total cells:      80 × 60 = 4800 cells
```

![Step 3.1: Cell Grid](images/hog_real_step3_1_cells.png)

## Step 3.3: 9-Bin Histogram Structure

**Parameters:**
- Range: 0° to 180° (unsigned)
- Bins: 9
- Bin width: 180° ÷ 9 = 20°

```
Bin   Range          Center
───   ─────────────  ──────
 0    [0°, 20°)       10°
 1    [20°, 40°)      30°
 2    [40°, 60°)      50°
 3    [60°, 80°)      70°
 4    [80°, 100°)     90°
 5    [100°, 120°)   110°
 6    [120°, 140°)   130°
 7    [140°, 160°)   150°
 8    [160°, 180°)   170°
```

![Step 3.3: Histogram Bins](images/hog_step3_3_histogram_bins.png)

## Step 3.4: Voting Process - Bilinear Interpolation

**Why bilinear interpolation?**
- Hard binning (all vote to one bin) causes aliasing
- Soft binning (vote split between bins) is smoother

**Formula:**

$$bin\_index = \frac{\theta}{20}$$

$$lower\_bin = floor(bin\_index) \mod 9$$
$$upper\_bin = (lower\_bin + 1) \mod 9$$

$$upper\_weight = bin\_index - floor(bin\_index)$$
$$lower\_weight = 1 - upper\_weight$$

$$Histogram[lower\_bin] += M \times lower\_weight$$
$$Histogram[upper\_bin] += M \times upper\_weight$$

### Example: Pixel with θ = 35°, M = 0.40

```
Step 1: bin_index = 35 / 20 = 1.75

Step 2: Find bins
  lower_bin = floor(1.75) = 1   → [20°, 40°)
  upper_bin = 2                 → [40°, 60°)

Step 3: Compute weights
  upper_weight = 1.75 - 1 = 0.75
  lower_weight = 1 - 0.75 = 0.25

Step 4: Add votes
  Histogram[1] += 0.40 × 0.25 = 0.10
  Histogram[2] += 0.40 × 0.75 = 0.30

Visual:
  θ = 35° is 75% toward bin 2 center from bin 1 center
  So 75% vote goes to bin 2, 25% to bin 1
```

![Step 3.4: Voting](images/hog_step3_4_voting.png)

![Cell Histogram Real](images/hog_real_step3_cell_histogram_real.png)

In [ ]:
def compute_cell_histograms(magnitude, direction, cell_size=8, num_bins=9):
    """
    Compute histogram of oriented gradients for each cell.
    
    Parameters:
    - magnitude: Gradient magnitude image
    - direction: Gradient direction image (0-180°)
    - cell_size: Size of each cell in pixels (default 8)
    - num_bins: Number of histogram bins (default 9 for 20° each)
    
    Returns:
    - histograms: Array of shape (cells_y, cells_x, num_bins)
    """
    h, w = magnitude.shape
    cells_y = h // cell_size
    cells_x = w // cell_size
    
    bin_width = 180.0 / num_bins  # 20 degrees per bin
    
    histograms = np.zeros((cells_y, cells_x, num_bins))
    
    for cy in range(cells_y):
        for cx in range(cells_x):
            y_start = cy * cell_size
            y_end = y_start + cell_size
            x_start = cx * cell_size
            x_end = x_start + cell_size
            
            cell_mag = magnitude[y_start:y_end, x_start:x_end]
            cell_dir = direction[y_start:y_end, x_start:x_end]
            
            hist = np.zeros(num_bins)
            
            for py in range(cell_size):
                for px in range(cell_size):
                    mag = cell_mag[py, px]
                    angle = cell_dir[py, px]
                    
                    # Bilinear interpolation
                    bin_idx = angle / bin_width
                    lower_bin = int(bin_idx) % num_bins
                    upper_bin = (lower_bin + 1) % num_bins
                    
                    upper_weight = bin_idx - int(bin_idx)
                    lower_weight = 1 - upper_weight
                    
                    hist[lower_bin] += mag * lower_weight
                    hist[upper_bin] += mag * upper_weight
            
            histograms[cy, cx, :] = hist
    
    return histograms

# Compute cell histograms
cell_size = 8
num_bins = 9
histograms = compute_cell_histograms(magnitude, direction, cell_size, num_bins)

cells_y, cells_x = histograms.shape[:2]
print(f"Cell size: {cell_size}×{cell_size} pixels")
print(f"Grid: {cells_y}×{cells_x} = {cells_y * cells_x} cells")
print(f"Bins per cell: {num_bins}")

---
# Step 4: Block Normalization

**Why normalize?** Different lighting produces different gradient magnitudes. Normalization makes HOG invariant to illumination.

## Step 4.1: Block Definition (2×2 cells)

```
Block = 2 × 2 cells = 16 × 16 pixels

Each cell has 9 histogram bins.
Block vector = [Cell(0,0) | Cell(1,0) | Cell(0,1) | Cell(1,1)]
             = [9 bins | 9 bins | 9 bins | 9 bins]
             = 4 × 9 = 36 values per block
```

```
┌─────────────┬─────────────┐
│  Cell (0,0) │  Cell (1,0) │
│   9 bins    │   9 bins    │
├─────────────┼─────────────┤
│  Cell (0,1) │  Cell (1,1) │
│   9 bins    │   9 bins    │
└─────────────┴─────────────┘

Block vector = 36 values
```

![Step 4.1: Block Definition](images/hog_step4_1_block_definition.png)

![Step 4.1: Block Layout](images/hog_real_step4_1_blocks.png)

## Step 4.2: 50% Block Overlap

**Why overlap?** Each cell contributes to multiple blocks, providing redundancy.

```
Stride = 1 cell (50% overlap)

cells_x = 80 cells
cells_y = 60 cells
block_size = 2 cells

blocks_x = 80 - 2 + 1 = 79
blocks_y = 60 - 2 + 1 = 59

Total blocks = 79 × 59 = 4661 blocks
```

```
Cells:     C0    C1    C2    C3    C4    ...
           ├────┴────┤
           ← Block 0 →
                 ├────┴────┤
                 ← Block 1 →

Each cell (except edges) appears in 4 different blocks!
```

![Step 4.2: Block Overlap](images/hog_step4_2_overlap.png)

## Step 4.3: L2 Normalization

**Formula:**

$$v_{normalized} = \frac{v}{\sqrt{||v||_2^2 + \epsilon^2}}$$

where:
- v = 36-element block vector
- $||v||_2 = \sqrt{\sum_i v_i^2}$ = L2 norm
- ε = 1e-6 (small constant for numerical stability)

### Illumination Invariance

**Key insight:** If brightness changes by factor k, gradients scale by k, but normalized vectors stay the same!

```
Original image: I(x,y)
Brighter image: I'(x,y) = k × I(x,y)  where k > 1

Gradients scale:
  Gx' = k × Gx
  Gy' = k × Gy

After L2 Normalization:
  H'_normalized = (k×H) / (k×||H||₂)
                = H / ||H||₂
                = H_normalized  ← SAME AS ORIGINAL!

CONCLUSION: L2 normalization cancels brightness scaling!
```

![Step 4.3: L2 Normalization](images/hog_step4_3_l2_normalization.png)

![Normalization Effect](images/hog_real_step4_2_normalized.png)

![Illumination Invariance](images/hog_illumination_invariance.png)

In [ ]:
def block_normalize(histograms, block_size=2, eps=1e-6):
    """
    Normalize histograms in overlapping blocks.
    
    L2 Normalization:
    v_normalized = v / √(||v||₂² + ε²)
    
    Parameters:
    - histograms: Cell histograms of shape (cells_y, cells_x, num_bins)
    - block_size: Number of cells per block dimension (default 2)
    - eps: Small constant for numerical stability
    
    Returns:
    - block_features: Normalized block vectors
    - block_shape: (blocks_y, blocks_x)
    """
    cells_y, cells_x, num_bins = histograms.shape
    
    blocks_y = cells_y - block_size + 1
    blocks_x = cells_x - block_size + 1
    
    block_features = []
    
    for by in range(blocks_y):
        for bx in range(blocks_x):
            block = histograms[by:by+block_size, bx:bx+block_size, :].flatten()
            
            # L2 normalization
            norm = np.sqrt(np.sum(block**2) + eps**2)
            block_normalized = block / norm
            
            block_features.append(block_normalized)
    
    return np.array(block_features), (blocks_y, blocks_x)

# Normalize blocks
block_features, block_shape = block_normalize(histograms, block_size=2)
blocks_y, blocks_x = block_shape

print(f"Block size: 2×2 cells")
print(f"Blocks: {blocks_y}×{blocks_x} = {blocks_y * blocks_x}")
print(f"Features per block: 4 cells × 9 bins = {block_features.shape[1]}")

## Step 4.5: Final Descriptor Assembly

```
Total blocks: 79 × 59 = 4661 blocks
Values per block: 36

HOG Descriptor = [Block(0,0) | Block(1,0) | ... | Block(78,58)]
               = 4661 × 36 = 167,796 dimensions
```

![Step 4.5: Final Descriptor](images/hog_real_step4_3_descriptor.png)

In [ ]:
# Final descriptor
descriptor = block_features.flatten()

print("=" * 60)
print("HOG DESCRIPTOR COMPUTED")
print("=" * 60)
print(f"Image size:        {W} × {H}")
print(f"Cell size:         {cell_size} × {cell_size} pixels")
print(f"Cells:             {cells_x} × {cells_y} = {cells_x * cells_y}")
print(f"Block size:        2 × 2 cells")
print(f"Blocks:            {blocks_x} × {blocks_y} = {blocks_x * blocks_y}")
print(f"Bins per cell:     {num_bins}")
print(f"Features/block:    36")
print(f"\nTotal descriptor:  {len(descriptor)}-D")
print("=" * 60)

---
# Complete Pipeline Summary

```
HOG ALGORITHM COMPLETE PIPELINE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

INPUT: RGB Image (640 × 480 × 3 = 921,600 values)
       ↓
STEP 1: Preprocessing
       • Grayscale: 640 × 480 = 307,200 values
       • Gamma correction: I^0.5
       ↓
STEP 2: Gradient Computation
       • Gx, Gy at each pixel
       • M = √(Gx² + Gy²)
       • θ = arctan(Gy/Gx) mod 180°
       • Output: 614,400 gradient values
       ↓
STEP 3: Cell Histograms
       • Divide into 8×8 cells: 80 × 60 = 4800 cells
       • 9-bin histogram per cell (bilinear interpolation)
       • Output: 43,200 histogram values
       ↓
STEP 4: Block Normalization
       • 2×2 blocks with 50% overlap: 79 × 59 = 4661 blocks
       • L2 normalization per block
       • Output: 167,796 normalized values
       ↓
OUTPUT: 167,796-D HOG Descriptor

COMPRESSION: 921,600 → 167,796 (5.5:1 ratio)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

![Complete HOG Pipeline](images/hog_real_complete_pipeline.png)

---
## Comparison: HOG vs SIFT

| Feature | HOG | SIFT |
|---------|-----|------|
| **Purpose** | Object detection | Feature matching |
| **Descriptor type** | Dense (entire window) | Sparse (keypoints) |
| **Dimension** | 167,796-D (for 640×480) | 128-D per keypoint |
| **Gradient bins** | 9 bins (0°-180°) | 8 bins (0°-360°) |
| **Scale invariance** | No (needs multi-scale) | Yes (built-in) |
| **Rotation invariance** | No | Yes |
| **Normalization** | L2 per 2×2 block | L2 per descriptor |
| **Use case** | "Is there a person?" | "Find this object" |

![Pedestrian Detection](images/hog_pedestrian_example.png)

## Mathematical Summary

### Derivative Formulas

| Formula | Description |
|---------|-------------|
| $G_x = I(x+1,y) - I(x-1,y)$ | Horizontal gradient (central difference) |
| $G_y = I(x,y+1) - I(x,y-1)$ | Vertical gradient (central difference) |
| $M = \sqrt{G_x^2 + G_y^2}$ | Gradient magnitude |
| $\theta = \arctan(G_y/G_x) \mod 180°$ | Gradient direction (unsigned) |

### Histogram Construction

| Formula | Description |
|---------|-------------|
| $bin\_width = 180° / 9 = 20°$ | Angular resolution |
| $bin\_idx = \theta / bin\_width$ | Continuous bin index |
| $lower\_weight = 1 - (bin\_idx - floor(bin\_idx))$ | Interpolation weight |
| $H[bin] += M \times weight$ | Vote accumulation |

### Block Normalization

| Formula | Description |
|---------|-------------|
| $||v||_2 = \sqrt{\sum_i v_i^2}$ | L2 norm |
| $v_{norm} = v / \sqrt{||v||_2^2 + \epsilon^2}$ | L2 normalization with stability |

![Math Summary](images/hog_math_summary.png)

In [ ]:
# Print final statistics
print("=" * 60)
print("HOG ALGORITHM COMPLETE")
print("=" * 60)
print(f"\nInput Image: {W} × {H} pixels")
print(f"\n1️⃣ PREPROCESSING:")
print(f"   Gamma correction (γ=0.5)")
print(f"\n2️⃣ GRADIENTS:")
print(f"   Magnitude range: [{magnitude.min():.4f}, {magnitude.max():.4f}]")
print(f"   Direction: 0°-180° (unsigned)")
print(f"\n3️⃣ CELL HISTOGRAMS:")
print(f"   Cell size: {cell_size}×{cell_size} pixels")
print(f"   Cells: {cells_x}×{cells_y} = {cells_x * cells_y}")
print(f"   Bins: {num_bins} (20° each)")
print(f"\n4️⃣ BLOCK NORMALIZATION:")
print(f"   Block size: 2×2 cells")
print(f"   Blocks: {blocks_x}×{blocks_y} = {blocks_x * blocks_y}")
print(f"   Features/block: 36")
print(f"\n5️⃣ FINAL DESCRIPTOR:")
print(f"   Dimension: {len(descriptor)}-D")
print("=" * 60)

---
## References

1. Dalal, N., & Triggs, B. (2005). "Histograms of oriented gradients for human detection." CVPR 2005.
2. Felzenszwalb, P. F., et al. (2010). "Object detection with discriminatively trained part-based models." PAMI 2010.